# **Feature Engineering**

In [42]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [43]:
# Load the cleaned dataset
ecommerce_df = pd.read_csv('dataset/cleaned/ecommerce_cleaned.csv', parse_dates=['order_date'])

In [44]:
ecommerce_df.sample(5)

,order_date,order_year,order_month,is_weekend,customer_name,gender,age,customer_segment,country,order_status,category,sub_category,unit_price_usd,quantity,discount_percent,revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,delivery_days,shipping_country,rating,customer_loyalty_score,coupon_used,session_duration_min,pages_visited,abandoned_cart_before,fraud_risk_score,device_type,campaign_source,traffic_source
83850,2026-01-30 19:01:53.694485,2026,1,No,Dakota Gonzales,Male,19,Regular,Spain,Returned,Health,Supplements,143.34,2,25,215.01,24.83,11.55,PayPal,Next Day,22.57,3,Spain,4,55.20,No,16.20,7,Yes,36.60,Mobile,Organic,Direct
277257,2024-02-06 15:13:07.402939,2024,2,No,Jesse Castaneda,Male,23,Regular,United States,Returned,Electronics,Audio,46.77,3,5,133.29,66.06,49.56,Credit Card,Next Day,2.03,6,United States,5,71.30,No,40.30,19,Yes,55.10,Tablet,Facebook,Social
767939,2025-11-09 22:39:57.960322,2025,11,Yes,Jacqueline Kelly,Female,19,Regular,Australia,Returned,Clothing,Womens Wear,60.37,2,20,96.59,37.85,39.19,Bank Transfer,Express,12.03,6,Australia,5,93.50,No,36.20,3,No,92.30,Mobile,Affiliate,Referral
441701,2024-06-03 19:23:44.922471,2024,6,No,Jeffrey Nelson,Male,19,Regular,Belgium,Returned,Electronics,Accessories,158.09,5,15,671.88,238.73,35.53,Bank Transfer,Standard,7.98,6,Belgium,2,19.60,No,41.30,9,No,74.80,Desktop,Email,Search
428490,2024-10-26 18:32:01.693429,2024,10,Yes,Erin Barron,Female,60,Regular,Canada,Completed,Health,Fitness,186.66,2,0,373.32,205.82,55.13,Apple Pay,Express,9.18,2,Canada,5,70.30,No,46.30,15,No,0.00,Mobile,Instagram,Referral


## Time

In [45]:
# Column for day of the week
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('order_month')+ 1,
    'order_day_of_week',
    ecommerce_df['order_date'].dt.strftime('%a'),
)

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('order_day_of_week') + 1,
    'part_day',
    pd.cut(
        ecommerce_df['order_date'].dt.hour, 
        bins=[0, 6, 12, 18, 24], 
        labels=['Night', 'Morning', 'Afternoon', 'Evening'], 
        right=False
    )
)

## Financial / Revenue

In [46]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('quantity') + 1,
    'gross_revenue_usd',
    ecommerce_df.eval('quantity * unit_price_usd')
)

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('gross_revenue_usd') + 1,
    'discount_amount_usd',
    ecommerce_df.eval('gross_revenue_usd * discount_percent / 100')
)

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('shipping_cost_usd') + 1,
    'shipping_cost_percent',
    ecommerce_df.eval('shipping_cost_usd / revenue_usd * 100')
)

## Discount Features

In [47]:
print(f'Minimum and Maximum Discount Percent: {ecommerce_df["discount_percent"].min()} - {ecommerce_df["discount_percent"].max()}')

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('discount_percent') + 1,
    'discount_tier',
    pd.cut(
        ecommerce_df['discount_percent'], 
        bins=[-1, 0, 10, 20, 25], 
        labels=['No Discount', 'Low Discount', 'Medium Discount', 'High Discount']
    )
)

Minimum and Maximum Discount Percent: 0 - 25


In [48]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('discount_tier') + 1,
    'is_discounted',
    ecommerce_df.eval('discount_percent > 0')
)

## Order/Basket Features

In [49]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('revenue_usd'),
    'order_value_segment',
    pd.qcut(
        ecommerce_df["revenue_usd"],
        q=4,
        labels=["Low", "Medium", "High", "Very High"]
    )
)

In [50]:
print(f'Minimum and Maximum Quantity {ecommerce_df["quantity"].min()} - {ecommerce_df["quantity"].max()}')

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('quantity'),
    'quantity_segment',
    pd.cut(
        ecommerce_df["quantity"],
        bins=[0, 1, 3, 6],
        labels=["Single", "Small Basket", "Bulk"]
    )
)

Minimum and Maximum Quantity 1 - 5


## Shipping / Logistics Features

In [51]:
print(f'Minimum and Maximum Delivery Days {ecommerce_df["delivery_days"].min()} - {ecommerce_df["delivery_days"].max()}')

ecommerce_df.insert(
    ecommerce_df.columns.get_loc('delivery_days') + 1,
    'delivery_speed',
    pd.cut(
        ecommerce_df["delivery_days"],
        bins=[0, 3, 7, np.inf],
        labels=["Fast", "Standard", "Slow"]
    )
)

Minimum and Maximum Delivery Days 1 - 14


## Customer Features

In [52]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('age') + 1,
    'age_group',
    pd.cut(
        ecommerce_df["age"],
        bins=[0, 24, 34, 44, 54, 64, np.inf],
        labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
    )
)

In [53]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('customer_loyalty_score') + 1,
    'loyalty_tier',
    pd.qcut(
        ecommerce_df["customer_loyalty_score"],
        q=3,
        labels=["Low", "Medium", "High"]
    )
)

## Customer Behavior Features

In [54]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('session_duration_min') + 1,
    'engagement_level',
    pd.qcut(
        ecommerce_df["session_duration_min"],
        q=3,
        labels=["Low", "Medium", "High"]
    )
)

## Fraud / Risk Features

In [55]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('fraud_risk_score') + 1,
    'fraud_risk_level',
    pd.qcut(
        ecommerce_df['fraud_risk_score'],
        q=3,
        labels=["Low", "Medium", "High"],
    )
)

## Product Pricing Features

In [56]:
ecommerce_df.insert(
    ecommerce_df.columns.get_loc('unit_price_usd') + 1,
    'price_tier',
    pd.qcut(
        ecommerce_df["unit_price_usd"],
        q=4,
        labels=["Budget", "Mid-Range", "Premium", "Luxury"]
    )
)

In [57]:
ecommerce_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000123 entries, 0 to 1000122
Data columns (total 48 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   order_date              1000123 non-null  datetime64[ns]
 1   order_year              1000123 non-null  int64         
 2   order_month             1000123 non-null  int64         
 3   order_day_of_week       1000123 non-null  object        
 4   part_day                1000123 non-null  category      
 5   is_weekend              1000123 non-null  object        
 6   customer_name           1000123 non-null  object        
 7   gender                  1000123 non-null  object        
 8   age                     1000123 non-null  int64         
 9   age_group               1000123 non-null  category      
 10  customer_segment        1000123 non-null  object        
 11  country                 1000123 non-null  object        
 12  order_status  

In [58]:
ecommerce_df.sample(10)

,order_date,order_year,order_month,order_day_of_week,part_day,is_weekend,customer_name,gender,age,age_group,customer_segment,country,order_status,category,sub_category,unit_price_usd,price_tier,quantity_segment,quantity,gross_revenue_usd,discount_amount_usd,discount_percent,discount_tier,is_discounted,order_value_segment,revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,shipping_cost_percent,delivery_days,delivery_speed,shipping_country,rating,customer_loyalty_score,loyalty_tier,coupon_used,session_duration_min,engagement_level,pages_visited,abandoned_cart_before,fraud_risk_score,fraud_risk_level,device_type,campaign_source,traffic_source
290251,2024-08-02 20:41:29.176769,2024,8,Fri,Evening,No,Patrick Powers,Male,64,55-64,Regular,Netherlands,Returned,Sports,Gym Equipment,41.52,Budget,Small Basket,3,124.56,24.91,20,Medium Discount,True,Low,99.65,36.89,37.02,Apple Pay,Express,20.01,20.08,3,Fast,Netherlands,1,48.40,Medium,No,21.20,Medium,9,No,5.50,Low,Desktop,Instagram,Social
187136,2024-08-29 04:04:42.686331,2024,8,Thu,Night,No,Andrew Smith,Male,24,18-24,Regular,Australia,Completed,Clothing,Womens Wear,54.77,Budget,Small Basket,2,109.54,0.00,0,No Discount,False,Low,109.54,55.18,50.37,Credit Card,Next Day,3.33,3.04,7,Standard,Australia,5,10.30,Low,Yes,10.60,Low,1,Yes,86.20,High,Mobile,Email,Referral
323165,2025-11-30 20:55:28.368645,2025,11,Sun,Evening,Yes,Timothy Cardenas,Male,45,45-54,VIP,Italy,Completed,Clothing,Womens Wear,18.75,Budget,Small Basket,2,37.50,5.62,15,Medium Discount,True,Low,31.88,9.34,29.30,Debit Card,Express,17.54,55.02,11,Slow,Italy,2,33.80,Medium,Yes,59.10,High,4,Yes,96.60,High,Desktop,Facebook,Email
968054,2025-04-30 07:02:52.601530,2025,4,Wed,Morning,No,Kenneth Jackson,Male,27,25-34,Regular,Germany,Completed,Health,Fitness,145.89,Premium,Bulk,4,583.56,58.36,10,Low Discount,True,High,525.20,287.00,54.65,PayPal,Express,7.60,1.45,6,Standard,Germany,3,98.70,High,No,3.90,Low,6,Yes,90.60,High,Desktop,Affiliate,Email
898970,2025-01-04 02:53:54.163338,2025,1,Sat,Night,Yes,Amy Porter,Female,25,25-34,Regular,Netherlands,Completed,Home,Kitchen,239.17,Luxury,Small Basket,3,717.51,0.00,0,No Discount,False,Very High,717.51,323.79,45.13,Credit Card,Express,4.12,0.57,8,Slow,Netherlands,4,0.60,Low,Yes,12.10,Low,17,No,47.20,Medium,Mobile,Instagram,Direct
796606,2024-07-27 17:25:42.624711,2024,7,Sat,Afternoon,Yes,Robin Garcia,Female,34,25-34,Regular,Australia,Completed,Sports,Outdoor,48.32,Budget,Bulk,5,241.60,36.24,15,Medium Discount,True,Medium,205.36,92.46,45.02,Debit Card,Express,12.37,6.02,7,Standard,Australia,5,63.40,Medium,Yes,58.50,High,20,Yes,34.80,Medium,Mobile,Instagram,Direct
465240,2024-08-28 17:35:39.667025,2024,8,Wed,Afternoon,No,Lisa Thornton,Female,64,55-64,Regular,Belgium,Completed,Home,Appliances,272.55,Luxury,Small Basket,3,817.65,0.00,0,No Discount,False,Very High,817.65,373.92,45.73,Apple Pay,Next Day,4.59,0.56,1,Fast,Belgium,4,54.50,Medium,Yes,52.10,High,5,Yes,75.20,High,Tablet,Organic,Search
825367,2025-12-24 04:05:50.689807,2025,12,Wed,Night,No,Johnny Mora,Male,58,55-64,Regular,Belgium,Completed,Home,Bedding,98.84,Mid-Range,Single,1,98.84,0.00,0,No Discount,False,Low,98.84,34.38,34.78,PayPal,Next Day,7.77,7.86,3,Fast,Belgium,2,62.70,Medium,Yes,11.50,Low,7,No,25.50,Low,Desktop,Google Ads,Social
660007,2025-03-13 05:32:00.260947,2025,3,Thu,Night,No,Jake Mccoy,Male,60,55-64,Premium,United States,Completed,Home,Decor,55.25,Budget,Small Basket,3,165.75,8.29,5,Low Discount,True,Medium,157.46,86.66,55.04,Apple Pay,Standard,4.27,2.71,8,Slow,United States,5,62.10,Medium,Yes,16.60,Low,16,No,33.80,Medium,Desktop,Email,Direct
486198,2025-04-29 05:30:56.218148,2025,4,Tue,Night,No,Tina Nash,Female,28,25-34,Premium,Netherlands,Completed,Sports,Gym Equipment,238.05,Luxury,Small Basket,2,476.10,71.42,15,Medium Discount,True,High,404.68,201.86,49.88,PayPal,Standard,19.72,4.87,13,Slow,Netherlands,1,89.50,High,Yes,5.10,Low,20,Yes,28.90,Low,Mobile,Facebook,Social


## Save Engineered/Processed Dataset

In [59]:
ecommerce_df.to_csv("dataset/processed/ecommerce_processed.csv")